In [1]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

MCP_TIME_URL = 'http://127.0.0.1:8325/mcp'

class StreamableHttpMcpToolSource:
    def __init__(self, url: str) -> None:
        self.url = url

    async def list_tools(self):
        async with streamablehttp_client(self.url) as (read, write, _):
            async with ClientSession(read, write) as session:
                await session.initialize()
                return await session.list_tools()

    async def call_tool(self, name: str, arguments: dict | None = None):
        async with streamablehttp_client(self.url) as (read, write, _):
            async with ClientSession(read, write) as session:
                await session.initialize()
                return await session.call_tool(name, arguments or {})

mcp_time_client = StreamableHttpMcpToolSource(MCP_TIME_URL)

mcp_tools = await mcp_time_client.list_tools()
tool_names = [tool.name for tool in mcp_tools.tools]
print('MCP tools:', tool_names)
if 'get_current_date_time' not in tool_names:
    raise RuntimeError('MCP server did not expose get_current_date_time')

direct_time = await mcp_time_client.call_tool('get_current_date_time', {})
print('direct get_current_date_time:', getattr(direct_time, 'structuredContent', None) or direct_time)


MCP tools: ['get_current_date_time', 'fetch_and_convert', 'search_web']
direct get_current_date_time: {'result': '2026-07-02 02:00:28'}


In [2]:
from openai_helpers_piplines_package import (
    JsonFixPipeline,
    LoggerPipeline,
    LoopGuardPipeline,
    PipelineRequestError,
    ToolPipeline,
    chat_session,
    classify_pipeline_error,
    dict_to_pydantic_schema,
    pipelined_chat,
    attempt,
)
import os
LOG_PATH = 'demo_pipeline.log'
BASE_URL = 'http://127.0.0.1:8080/v1'
API_KEY = "none"
MODEL_NAME = ""


In [3]:
from openai import OpenAI
import json
from dataclasses import asdict

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

chat = pipelined_chat(
    client.chat.completions,
    layers=[LoggerPipeline(path=LOG_PATH), ToolPipeline(max_retries=3),LoopGuardPipeline(max_retries=3)],
)


In [4]:

session = chat_session(
    chat,
    model=MODEL_NAME,
    role_content={'system': 'Use the MCP-tools when it possible.',},
    temperature=0.1,
    max_tokens=500,
    tool_sources=[mcp_time_client],
    return_trace=True,
)

mcp_result = await attempt(session.step(role_content={'user':'tell current date and time and current news from CNN'}))


if isinstance(mcp_result, PipelineRequestError):
    print('MCP tool demo failed:')
    print(mcp_result)
else:
    print('assistant:', mcp_result.response['choices'][0]['message'].get('content', ''))
    print('tool executions:')
    print(json.dumps([asdict(execution) for execution in mcp_result.tool_executions], indent=2, ensure_ascii=False))
    print('trace events:', [f'{event.level}/{event.action}' for event in (mcp_result.trace or [])])

assistant: The current date and time is **2026-07-02 02:00:37**.

Here are the top news headlines from CNN:

*   **Politics & World:**
    *   Qatari leaders meet senior US and Iranian officials to advance talks.
    *   China tells its minorities to integrate or face consequences with a sweeping new law.
    *   Ukraine launches a major drone attack on Moscow, hitting a satellite center.
    *   Trump made over a billion dollars from cryptocurrency ventures in his first year back in office.
    *   Trump wants to ditch his signature trade deal, but it's not that easy.
    *   Crisis in Hormuz seen as a net win for China for now.
    *   Pope Leo faces his first major crisis of papacy from a rebel Catholic group.

*   **US News:**
    *   Fast-moving wildfires scorch thousands of acres across the Western US.
    *   Trump says he told Bill Pulte to 'declassify almost everything'.
    *   Trump claims a glorious win streak, but history is keeping score.
    *   Trump’s audacious bid to 

In [7]:
print(mcp_result.response['choices'][0]['message'].get('reasoning_content', ''))


The user asked for the current date and time and current news from CNN.
I have successfully retrieved the current date and time: "2026-07-02 02:00:37".
I have also successfully fetched the top news headlines from CNN.

Now I need to present this information clearly to the user.
I will state the date and time first, then list the top news headlines.
The headlines cover a wide range of topics: politics (Trump, US/Iran talks), international news (China, Greece, Venezuela, Ukraine), sports (World Cup, Serena Williams), science (oceans, digital camera, cell from scratch), and other events (wildfires, deaths).

I will format the response nicely.



In [ ]:
stop()



In [15]:
session = chat_session(
    chat,
    model=MODEL_NAME,
    role_content={'system': 'Use the MCP-tools when it possible',},
    temperature=0.1,
    max_tokens=500,
    tool_sources=[mcp_time_client],
    return_trace=True,
)

mcp_result = await attempt(session.step(role_content={'user':'what benchmarks for LLM menthal/abstraction testing exist?'}))


if isinstance(mcp_result, PipelineRequestError):
    print('MCP tool demo failed:')
    print(mcp_result)
else:
    print('assistant:', mcp_result.response['choices'][0]['message'].get('content', ''))
    print('tool executions:')
    print(json.dumps([asdict(execution) for execution in mcp_result.tool_executions], indent=2, ensure_ascii=False))
    print('trace events:', [f'{event.level}/{event.action}' for event in (mcp_result.trace or [])])



AttributeError: 'ToolIterationLimitExceededError' object has no attribute 'response'

In [ ]:
mcp_result = await attempt(session.step(role_content={'user':'tell detailes about Germany Shooting. How police acts?'}))

if isinstance(mcp_result, PipelineRequestError):
    print('MCP tool demo failed:')
    print(mcp_result)
else:
    print('assistant:', mcp_result.response['choices'][0]['message'].get('content', ''))
    print('tool executions:')
    print(json.dumps([asdict(execution) for execution in mcp_result.tool_executions], indent=2, ensure_ascii=False))
    print('trace events:', [f'{event.level}/{event.action}' for event in (mcp_result.trace or [])])

MCP tool demo failed:
chat.completions.create failed: Loading model

Request stage: tool_generation

Request params: model='', temperature=0.1, max_tokens=500, stream=True, stream_options={'include_usage': True}, tools=['get_current_date_time', 'fetch_and_convert', 'search_web']
Messages (8):
  [0] system: 'Use the MCP-tools when it possible.'
  [1] user: 'tell current date and time and current news from CNN'
  [2] assistant:
        The current date and time is **2026-07-02 01:25:24**.
        
        Here are the top news stories from CNN:
        
        **World & Politics**
        *   **US-Iran Talks:** Qatari leaders are meeting with senior US and Iranian officials in an effort to advance talks.
        *   **China's New Law:** China has introduced a sweeping new law telling its ethnic minorities to integrate or face consequences.
        *   **Greece Violence:** One person is dead and several others are hurt after firebombs were hurled at the homes of members of Greece's gover

In [12]:
session = chat_session(
    chat,
    model=MODEL_NAME,
    role_content={'system': 'what benchmarks for LLM menthal/abstraction testing exist?',},
    temperature=0.1,
    max_tokens=500,
    tool_sources=[mcp_time_client],
    return_trace=True,
)

mcp_result = await attempt(session.step(role_content={'user':'compare last news from CNN and BBC'}))



if isinstance(mcp_result, PipelineRequestError):
    print('MCP tool demo failed:')
    print(mcp_result)
else:
    print('assistant:', mcp_result.response['choices'][0]['message'].get('content', ''))
    print('tool executions:')
    print(json.dumps([asdict(execution) for execution in mcp_result.tool_executions], indent=2, ensure_ascii=False))
    print('trace events:', [f'{event.level}/{event.action}' for event in (mcp_result.trace or [])])

assistant: Based on the latest headlines from CNN and BBC (dated around July 2, 2026), here is a comparison of the news coverage.

### **Shared Headlines**
Both outlets are covering several major global events, though with different angles:

*   **Venezuela Earthquake:** Both report on the "miraculous" rescue of a man (Hernán Gil) trapped under rubble for eight days.
*   **Taylor Swift & Travis Kelce Wedding:** Both are covering the "multi-day wedding celebration" taking place in New York, with CNN providing live updates and BBC focusing on the star-studded guest list and fashion.
*   **FIFA World Cup 2026:** Both cover the Round of 32/16 matches. CNN highlights the "USA’s red card controversy" and Portugal's win, while BBC focuses on Portugal's chaotic win and Switzerland's match against Algeria.
*   **Monaco Bombing:** Both report on the bomb attack in Monaco. CNN discusses how it "upended the reputation" of the safe haven, while BBC reports that an arrest warrant has been issued for